In [1]:
"""this code is written by Mohamed Thaufeek for the purpose of building an ML model to correct HTML code"""

import pandas as pd


#Load dataset to pandas data frame from csv file
Data=pd.read_csv('html_correction_dataset.csv')
BadCode=Data['bad_code'].astype(str).values
GoodCode=Data['good_code'].astype(str).values
print(Data.head())


ModuleNotFoundError: No module named 'pandas'

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

#Tokenization
Tokenizer=Tokenizer(filters='', lower=False, oov_token='<OOV>')
Tokenizer.fit_on_texts(np.concatenate((BadCode, GoodCode)))  #Fit tokenizer on both

#Convert texts to sequences
BadCodeSeq=Tokenizer.texts_to_sequences(BadCode)
GoodCodeSeq=Tokenizer.texts_to_sequences(GoodCode)

#Padding sequences to the maximum sequence length
MaxSequenceLength=max(max(len(seq) for seq in BadCodeSeq), max(len(seq) for seq in GoodCodeSeq))
BadCodePadded=pad_sequences(BadCodeSeq, maxlen=MaxSequenceLength, padding='post')
GoodCodePadded=pad_sequences(GoodCodeSeq, maxlen=MaxSequenceLength, padding='post')

#Print padded sequences to verify
print(BadCodePadded[0])
print(GoodCodePadded[0])

In [ ]:
from sklearn.model_selection import train_test_split

#Splitting data into train and test sets
XTrain, XTest, YTrain, YTest=train_test_split(BadCodePadded, GoodCodePadded, test_size=0.2, random_state=42)

#Print shapes to verify
print(XTrain.shape, YTrain.shape)
print(XTest.shape, YTest.shape)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

#Building the model
VocabSize=len(Tokenizer.word_index) + 1  #Vocabulary size

Model=Sequential([
    Embedding(VocabSize, 128, input_length=MaxSequenceLength),
    LSTM(128, return_sequences=True, dropout=0.2),
    LSTM(128, return_sequences=True, dropout=0.2),
    Dense(VocabSize, activation='softmax')  #Output layer with VocabSize classes
])

#Force the model to build by running the model on a dummy input
Model.build(input_shape=(None, MaxSequenceLength))  #(None, MaxSequenceLength) means batch size is flexible
Model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

#Print the model summary
Model.summary()



In [ ]:
from sklearn.model_selection import train_test_split

#Splitting data into train and test sets
X_train, X_test, y_train, y_test=train_test_split(BadCodePadded, GoodCodePadded, test_size=0.2, random_state=42)

#Print shapes to verify
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

#Define vocab_size and max_sequence_length
vocab_size=len(Tokenizer.word_index) + 1  #vocab_size = total unique words in the tokenizer
max_sequence_length = 100  #Example length, replace with your actual max length

#Define the model
model=Sequential([
    Embedding(vocab_size, 128, input_length=max_sequence_length),  
    LSTM(128, return_sequences=True, dropout=0.2), 
    LSTM(128, return_sequences=True, dropout=0.2), 
    Dense(vocab_size, activation='softmax')  #Assuming a classification task
])

#Build the model with an explicit input shape
model.build(input_shape=(None, max_sequence_length))  #(None, max_sequence_length) for variable batch size
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

#Print the model summary
model.summary()

In [ ]:
#Training data 
y_train_shifted=np.expand_dims(y_train, axis=-1)
y_test_shifted=np.expand_dims(y_test, axis=-1)

#Check the shape of shifted targets
print(y_train_shifted.shape, y_test_shifted.shape)



In [ ]:
#Train the model
History=Model.fit(XTrain, YTrain, validation_data=(XTest, YTest), epochs=500, batch_size=64)

#Monitor training history
print(History.history['accuracy'][-1], History.history['val_accuracy'][-1])

#Save the above model as h5 file
Model.save('model.h5')

In [ ]:
def CorrectHTMLCode(bad_code_sample):
    #Tokenize and pad the input
    InputSequence=Tokenizer.texts_to_sequences([bad_code_sample])
    InputPadded=pad_sequences(InputSequence, maxlen=max_sequence_length, padding='post')

    #Predict the output sequence
    PredictedSequence=model.predict(InputPadded)
    PredictedIndices=np.argmax(PredictedSequence, axis=-1)

    #Convert indices back to text
    CorrectedHTML=' '.join([Tokenizer.index_word[idx] for idx in PredictedIndices[0] if idx != 0])

    return CorrectedHTML

#Example usage
SampleBadCode="""<!DOCTYPE html>
<html>
<head>
    <title>Gallery</title>
</head>
<body>
    <h1>Photo Gallery</h1>
    <img src="photo1.jpg">
    <img src="photo2.jpg">
</body>
</html>

"""
CorrectedHTML=CorrectHTMLCode(SampleBadCode)

#Display the results
print("Original Bad Code:", SampleBadCode)
print("Corrected HTML Code:", CorrectedHTML)
